In [1]:
import stim
import itertools
import numpy as np
import pickle
import time
import os
import re
import pprint
import numpy as np

from autqec.automorphisms   import *
from autqec.utils.qec       import *
from autqec.utils.qiskit    import *
from autqec.graph_auts      import *
from autqec.ZX_dualities    import *
from autqec.ZY_dualities    import *
from autqec.magma_interface import *
from autqec.code_embedding  import *

from magma_online           import *
from stim_helpers           import *
from compare_circuits       import *
from cws_helpers            import *

### 5-qubit code

In [2]:
generators_5qubits = [
    stim.PauliString("XZZXI"),
    stim.PauliString("IXZZX"),
    stim.PauliString("XIXZZ"),
    stim.PauliString("ZZXIX")
]

# logical operators
logical_I = stim.PauliString("IIIII")
logical_X = stim.PauliString("XXXXX")
logical_Z = stim.PauliString("ZZZZZ")
logical_Y = canonicalize_real_pauli(logical_X * logical_Z)

logical_X_min = stim.PauliString("-YIXIY")
logical_Z_min = stim.PauliString("-XIZIX")
logical_Y_min = canonicalize_real_pauli(logical_X_min * logical_Z_min)

logical_basis5 = {
    "I": logical_I,
    "X": canonicalize_real_pauli(logical_X_min),
    "Y": canonicalize_real_pauli(logical_Y_min),
    "Z": canonicalize_real_pauli(logical_Z_min)
}

### [[10,2,3]] stabilizer generators: 2 independent copies of [[5,1,3]]

In [3]:
generators_5qubits_2blocks = []

for s in generators_5qubits:
    generators_5qubits_2blocks.append(s + logical_I)
    generators_5qubits_2blocks.append(logical_I + s)

# single-block logicals on each encoded qubit
logical_XI = logical_X_min + logical_I
logical_ZI = logical_Z_min + logical_I

logical_IX = logical_I + logical_X_min
logical_IZ = logical_I + logical_Z_min

logical_YI = canonicalize_real_pauli(logical_XI * logical_ZI)
logical_IY = canonicalize_real_pauli(logical_IX * logical_IZ)

# convenient logical basis for generators
logical_basis_2_generators = {
    "XI": canonicalize_real_pauli(logical_XI),
    "YI": canonicalize_real_pauli(logical_YI),
    "ZI": canonicalize_real_pauli(logical_ZI),

    "IX": canonicalize_real_pauli(logical_IX),
    "IY": canonicalize_real_pauli(logical_IY),
    "IZ": canonicalize_real_pauli(logical_IZ),
}

# full 2-qubit Pauli logical basis, excluding II
logical_basis_2blocks = {}

for a in ["I", "X", "Y", "Z"]:
    for b in ["I", "X", "Y", "Z"]:
        label = a + b
        if label == "II":
            continue
        pauli = (
              (logical_basis5[a] if a != "I" else logical_I)
            + (logical_basis5[b] if b != "I" else logical_I)
        )
        logical_basis_2blocks[label] = canonicalize_real_pauli(pauli)

print_combined_stabilizers(
    generators_5qubits_2blocks,
    "Combined generators: two blocks of [[5,1,3]] -> [[10,2,3]]",
    num_blocks=2
)

print("Number of stabilizer generators: ",  len(generators_5qubits_2blocks))
print("Number of logical basis elements:", len(logical_basis_2blocks))


Combined generators: two blocks of [[5,1,3]] -> [[10,2,3]]

+XZZXIIIIII   +IIIIIXZZXI
+IXZZXIIIII   +IIIIIIXZZX
+XIXZZIIIII   +IIIIIXIXZZ
+ZZXIXIIIII   +IIIIIZZXIX

Number of stabilizer generators:  8
Number of logical basis elements: 15


### Prologue, Epilogue, and CZ round-robin circuits

In [4]:
# Prologue: single-qubit Clifford layer (provided)
prologue_2blocks = stim.Circuit()
prologue_2blocks.append("I", [i for i in range(10)])
prologue_2blocks.append("H", [0])
prologue_2blocks.append("S", [0])
prologue_2blocks.append("Y", [2])
prologue_2blocks.append("H", [4])
prologue_2blocks.append("S", [4])
prologue_2blocks.append("H", [5])
prologue_2blocks.append("S", [5])
prologue_2blocks.append("Y", [7])
prologue_2blocks.append("H", [9])
prologue_2blocks.append("S", [9])

# Epilogue: inverse single-qubit layer (provided)
epilogue_2blocks = stim.Circuit()
epilogue_2blocks.append("S", [0])
epilogue_2blocks.append("S", [0])
epilogue_2blocks.append("S", [0])
epilogue_2blocks.append("H", [0])
epilogue_2blocks.append("Y", [2])
epilogue_2blocks.append("S", [4])
epilogue_2blocks.append("S", [4])
epilogue_2blocks.append("S", [4])
epilogue_2blocks.append("H", [4])
epilogue_2blocks.append("S", [5])
epilogue_2blocks.append("S", [5])
epilogue_2blocks.append("S", [5])
epilogue_2blocks.append("H", [5])
epilogue_2blocks.append("Y", [7])
epilogue_2blocks.append("S", [9])
epilogue_2blocks.append("S", [9])
epilogue_2blocks.append("S", [9])
epilogue_2blocks.append("H", [9])

# Round-robin part 1: 6 CZ gates (cross-block)
# Each is a CZ between a qubit in block 0 {0,2,4} and block 1 {5,7,9}
RR_circuit_p1 = [
    ("CZ", [0, 5]),
    ("CZ", [2, 7]),
    ("CZ", [4, 9]),
    ("CZ", [0, 9]),
    ("CZ", [2, 5]),
    ("CZ", [4, 7]),
]

# Round-robin part 2: 3 CZ gates (cross-block)
RR_circuit_p2 = [
    ("CZ", [0, 7]),
    ("CZ", [2, 9]),
    ("CZ", [4, 5]),
]

print("Prologue circuit:")
print(prologue_2blocks)
print("\nEpilogue circuit:")
print(epilogue_2blocks)

Prologue circuit:
I 0 1 2 3 4 5 6 7 8 9
H 0
S 0
Y 2
H 4
S 4
H 5
S 5
Y 7
H 9
S 9

Epilogue circuit:
S 0 0 0
H 0
Y 2
S 4 4 4
H 4
S 5 5 5
H 5
Y 7
S 9 9 9
H 9


In [5]:
def update_logical_basis_by_circuit(logical_basis: dict, circuit: stim.Circuit) -> dict:
    """
    Conjugate every logical operator in a basis dict by a Stim circuit.
    Returns a new dict with the same keys and updated PauliStrings.
    """
    return {
        label: canonicalize_real_pauli(conjugate_stabilizer_by_circuit(op, circuit))
        for label, op in logical_basis.items()
    }

### Code after prologue

In [6]:
generators_5qubits_2blocks_after_prologue = [
    conjugate_stabilizer_by_circuit(s, prologue_2blocks)
    for s in generators_5qubits_2blocks
]

print_combined_stabilizers(
    generators_5qubits_2blocks_after_prologue,
    "Combined generators after prologue: two blocks of [[5,1,3]] -> [[10,2,3]]",
    num_blocks=2
)

logical_basis_2blocks_after_prologue = update_logical_basis_by_circuit(
    logical_basis_2blocks, prologue_2blocks
)


Combined generators after prologue: two blocks of [[5,1,3]] -> [[10,2,3]]

-ZZZXIIIIII   -IIIIIZZZXI
-IXZZZIIIII   -IIIIIIXZZZ
-ZIXZYIIIII   -IIIIIZIXZY
-YZXIZIIIII   -IIIIIYZXIZ



### Converting to CWS Code

In [7]:
graph_stabs, C, A, hadamard_qubits = stabilizer_code_to_cws(
    stabilizers = generators_5qubits_2blocks_after_prologue,
    logical_Zs  = [logical_basis_2blocks_after_prologue["IZ"],
                   logical_basis_2blocks_after_prologue["ZI"]],
    logical_Xs  = [logical_basis_2blocks_after_prologue["IX"],
                   logical_basis_2blocks_after_prologue["XI"]],
)

print("Graph stabilizers after prologue:")
print("Standard Form:")
for g in graph_stabs:
    print(g)

print("\nCode Words:")
for c in C:
    print(c)

print("\nAdj Matrix =\n", A)

Graph stabilizers after prologue:
Standard Form:
+XZ_ZZ_____
+ZX_Z______
+__XZZ_____
+ZZZX______
+Z_Z_X_____
+_____XZ_ZZ
+_____ZX_Z_
+_______XZZ
+_____ZZZX_
+_____Z_Z_X

Code Words:
0000000000
0000001001
0100100000
0100101001

Adj Matrix =
 [[0 1 0 1 1 0 0 0 0 0]
 [1 0 0 1 0 0 0 0 0 0]
 [0 0 0 1 1 0 0 0 0 0]
 [1 1 1 0 0 0 0 0 0 0]
 [1 0 1 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 0 1 1]
 [0 0 0 0 0 1 0 0 1 0]
 [0 0 0 0 0 0 0 0 1 1]
 [0 0 0 0 0 1 1 1 0 0]
 [0 0 0 0 0 1 0 1 0 0]]


### Apply CZ round-robin circuits

Note: CZ gates are Clifford, so they update the graph (2-edges) directly via conjugation — no hyperedges are introduced. We track their effect on A separately.

In [8]:
def apply_cz_to_graph(A, cz_gates):
    """
    Apply a list of CZ gates to the graph adjacency matrix A (GF2 toggle).
    CZ(a,b) toggles the edge {a,b} in the graph.
    Returns updated adjacency matrix and resulting hyperedges dict.
    """
    A_new = A.copy()
    for gate, qubits in cz_gates:
        if gate != "CZ":
            raise ValueError(f"Expected CZ, got {gate}")
        a, b = qubits
        A_new[a, b] ^= 1
        A_new[b, a] ^= 1
    # Build hyperedges from 2-edges only (no 3-edges from CZ)
    n = A_new.shape[0]
    hyperedges = {}
    for i in range(n):
        for j in range(i+1, n):
            if A_new[i, j]:
                hyperedges[frozenset({i, j})] = 1
    return A_new, hyperedges


# Part 1: 6 CZ gates
A_p1, hyperedges_p1 = apply_cz_to_graph(A, RR_circuit_p1)
C_p1 = list(C)  # CZ is Clifford: codewords unchanged up to stabilizer

print("After part-1 CZ round-robin:")
print_hypergraph(hyperedges_p1)
print("\nUpdated adj matrix:\n", A_p1)

# Part 2: 3 CZ gates (applied on top of part 1)
A_p2, hyperedges_p2 = apply_cz_to_graph(A_p1, RR_circuit_p2)
C_p2 = list(C)

print("\nAfter part-2 CZ round-robin (cumulative):")
print_hypergraph(hyperedges_p2)

After part-1 CZ round-robin:
  2-edges (18): [[0, 1], [0, 3], [0, 4], [0, 5], [0, 9], [1, 3], [2, 3], [2, 4], [2, 5], [2, 7], [4, 7], [4, 9], [5, 6], [5, 8], [5, 9], [6, 8], [7, 8], [7, 9]]
  3-edges (0): []

Updated adj matrix:
 [[0 1 0 1 1 1 0 0 0 1]
 [1 0 0 1 0 0 0 0 0 0]
 [0 0 0 1 1 1 0 1 0 0]
 [1 1 1 0 0 0 0 0 0 0]
 [1 0 1 0 0 0 0 1 0 1]
 [1 0 1 0 0 0 1 0 1 1]
 [0 0 0 0 0 1 0 0 1 0]
 [0 0 1 0 1 0 0 0 1 1]
 [0 0 0 0 0 1 1 1 0 0]
 [1 0 0 0 1 1 0 1 0 0]]

After part-2 CZ round-robin (cumulative):
  2-edges (21): [[0, 1], [0, 3], [0, 4], [0, 5], [0, 7], [0, 9], [1, 3], [2, 3], [2, 4], [2, 5], [2, 7], [2, 9], [4, 5], [4, 7], [4, 9], [5, 6], [5, 8], [5, 9], [6, 8], [7, 8], [7, 9]]
  3-edges (0): []


### Automorphism group via Magma

In [9]:
n = 10
k = 2
d = 3

command_file_name = f"./magma_commands_n{n}k{k}d{d}.txt"
output_file_name  = f"./magma_output_n{n}k{k}d{d}.txt"

# Generator codewords: C[1] = |01>, C[2] = |10>
C_new_gens = [C[1], C[2]]

magma_lines = []
magma_lines.append("F := GF(2);")
magma_lines.append("G := Matrix(F, [")
magma_lines += [
    "  [" + ",".join(row) + "]" + ("," if i < len(C_new_gens) - 1 else "")
    for i, row in enumerate(C_new_gens)
]
magma_lines.append("]);")
magma_lines.append("C := LinearCode(G);")
magma_lines.append("A_aut := AutomorphismGroup(C);")
magma_lines.append("A_aut;")
magma_lines.append("Generators(A_aut);")
magma_lines.append("#A_aut;")

with open(command_file_name, "w") as f:
    f.write("\n".join(magma_lines))

run_magma_online_from_file(command_file_name)

with open(output_file_name, "r") as f:
    magma_output = f.read()

print(magma_output)


def parse_magma_permutation_group(output_text, n):
    """
    Parse Magma permutation generators into Python 0-indexed permutations of length n.
    Skips the set literal { ... } block that Magma prints for Generators(A).
    """
    cleaned = re.sub(r'\{[^}]*\}', '', output_text, flags=re.DOTALL)

    perms = []
    for line in cleaned.splitlines():
        line = line.strip()
        if not line or not line.startswith('('):
            continue

        perm = list(range(n))
        cycles = re.findall(r'\(([^)]+)\)', line)
        for cyc in cycles:
            cyc_nums = [int(x) - 1 for x in cyc.split(',')]
            for i in range(len(cyc_nums)):
                perm[cyc_nums[i]] = cyc_nums[(i + 1) % len(cyc_nums)]
        perms.append(perm)

    return perms


auts = parse_magma_permutation_group(magma_output, n=n)
print("num generators =", len(auts))
for p in auts:
    print(p)

Permutation group A_aut acting on a set of cardinality 10
Order = 5760 = 2^7 * 3^2 * 5
(3, 4)
(2, 7)(5, 10)
(4, 6)
(2, 5)
(1, 9)
(1, 3)
(6, 8)
(7, 10)
{
(7, 10),
(3, 4),
(4, 6),
(2, 7)(5, 10),
(2, 5),
(1, 9),
(1, 3),
(6, 8)
}
5760

num generators = 8
[0, 1, 3, 2, 4, 5, 6, 7, 8, 9]
[0, 6, 2, 3, 9, 5, 1, 7, 8, 4]
[0, 1, 2, 5, 4, 3, 6, 7, 8, 9]
[0, 4, 2, 3, 1, 5, 6, 7, 8, 9]
[8, 1, 2, 3, 4, 5, 6, 7, 0, 9]
[2, 1, 0, 3, 4, 5, 6, 7, 8, 9]
[0, 1, 2, 3, 4, 7, 6, 5, 8, 9]
[0, 1, 2, 3, 4, 5, 9, 7, 8, 6]


### Hypergraph automorphism filtering

In [10]:
def check_codeword_preserving(perm, C_new):
    """Check if a permutation maps the codespace to itself."""
    C_set = set(C_new)
    for word in C_new:
        permuted = "".join(word[perm[i]] for i in range(len(perm)))
        if permuted not in C_set:
            return False
    return True


def check_hyperedge_preserving(perm, hyperedges):
    """Check if a permutation maps every hyperedge to another hyperedge."""
    for edge in hyperedges:
        permuted_edge = frozenset(perm[q] for q in edge)
        if permuted_edge not in hyperedges:
            return False
    return True


def filter_hypergraph_automorphisms(auts, C_new, hyperedges):
    """Filter automorphisms preserving both codewords and hypergraph."""
    valid = []
    for i, p in enumerate(auts):
        preserves_C = check_codeword_preserving(p, C_new)
        preserves_H = check_hyperedge_preserving(p, hyperedges)
        if preserves_C and preserves_H:
            print(f"gen {i:2d}: preserves C={preserves_C}  preserves H={preserves_H}")
            valid.append((i, p))
    return valid


def filter_hypergraph_automorphisms_by_order(auts, C_new, hyperedges):
    """Break down preservation by edge order for diagnosis."""
    edges_2 = {e for e in hyperedges if len(e) == 2}
    edges_3 = {e for e in hyperedges if len(e) == 3}
    for i, p in enumerate(auts):
        preserves_C = check_codeword_preserving(p, C_new)
        preserves_2 = all(frozenset(p[q] for q in e) in edges_2 for e in edges_2)
        preserves_3 = all(frozenset(p[q] for q in e) in edges_3 for e in edges_3)
        print(f"gen {i:2d}: C={preserves_C}  2-edges={preserves_2}  3-edges={preserves_3}")


# Check against part-1 hyperedges
print("=== Part 1 (6 CZ gates) ===")
filter_hypergraph_automorphisms_by_order(auts, C, hyperedges_p1)

print("\n=== Part 2 (6+3 CZ gates cumulative) ===")
filter_hypergraph_automorphisms_by_order(auts, C, hyperedges_p2)

=== Part 1 (6 CZ gates) ===
gen  0: C=True  2-edges=False  3-edges=True
gen  1: C=True  2-edges=False  3-edges=True
gen  2: C=True  2-edges=False  3-edges=True
gen  3: C=True  2-edges=False  3-edges=True
gen  4: C=True  2-edges=False  3-edges=True
gen  5: C=True  2-edges=False  3-edges=True
gen  6: C=True  2-edges=False  3-edges=True
gen  7: C=True  2-edges=False  3-edges=True

=== Part 2 (6+3 CZ gates cumulative) ===
gen  0: C=True  2-edges=False  3-edges=True
gen  1: C=True  2-edges=False  3-edges=True
gen  2: C=True  2-edges=False  3-edges=True
gen  3: C=True  2-edges=False  3-edges=True
gen  4: C=True  2-edges=False  3-edges=True
gen  5: C=True  2-edges=False  3-edges=True
gen  6: C=True  2-edges=False  3-edges=True
gen  7: C=True  2-edges=False  3-edges=True


### Group closure

In [11]:
def compose(p, q):
    """Compose permutations: p after q."""
    return [p[q[i]] for i in range(len(p))]


def invert_perm(p):
    inv = [0] * len(p)
    for i, j in enumerate(p):
        inv[j] = i
    return inv


def closure_limited(gens, n, max_size=200000):
    ident = tuple(range(n))
    seen = {ident}
    frontier = [list(ident)]
    gens2 = gens + [invert_perm(g) for g in gens]

    while frontier and len(seen) < max_size:
        cur = frontier.pop(0)  # BFS
        for g in gens2:
            nxt = tuple(compose(g, cur))
            if nxt not in seen:
                seen.add(nxt)
                frontier.append(list(nxt))

    return [list(p) for p in seen]


auts_closure = closure_limited(auts, n=n, max_size=100000)
print(f"Closure size: {len(auts_closure)}")

# Filter against both CZ hypergraphs
for label, hyp, C_cur in [
    ("Part 1 (6 CZ)", hyperedges_p1, C),
    ("Part 2 (6+3 CZ)", hyperedges_p2, C),
]:
    valid = filter_hypergraph_automorphisms(auts_closure, C_cur, hyp)
    print(f"\n{label}: {len(valid)} valid automorphisms")

Closure size: 5760
gen 692: preserves C=True  preserves H=True

Part 1 (6 CZ): 1 valid automorphisms
gen 513: preserves C=True  preserves H=True
gen 692: preserves C=True  preserves H=True

Part 2 (6+3 CZ): 2 valid automorphisms


### Edge-order analysis

In [12]:
def three_edges_only(hyperedges):
    return {e for e in hyperedges if len(e) == 3}

def two_edges_only(hyperedges):
    return {e for e in hyperedges if len(e) == 2}

def preserves_3_hyperedges(perm, hyperedges):
    H3 = three_edges_only(hyperedges)
    return {frozenset(perm[q] for q in e) for e in H3} == H3

def edge_symmetric_difference_after_perm(perm, hyperedges):
    E = two_edges_only(hyperedges)
    E_perm = {frozenset(perm[q] for q in e) for e in E}
    return E_perm ^ E

# For part-2 cumulative hyperedges (most constrained)
valid_3edge = [
    p for p in auts_closure
    if preserves_3_hyperedges(p, hyperedges_p2)
]

print(f"valid preserving 2-edge structure of part-2 = {len(valid_3edge)}")
for p in valid_3edge[:20]:
    diff = edge_symmetric_difference_after_perm(p, hyperedges_p2)
    print(f"perm: {p}")
    print(f"  2-edge mismatch size: {len(diff)}")
    print(f"  {sorted([sorted(e) for e in diff])}")
    print()

valid preserving 2-edge structure of part-2 = 5760
perm: [3, 6, 7, 0, 9, 2, 1, 8, 5, 4]
  2-edge mismatch size: 18
  [[0, 1], [0, 4], [0, 5], [0, 6], [0, 9], [1, 2], [1, 3], [1, 5], [3, 4], [3, 6], [3, 8], [3, 9], [4, 5], [4, 8], [5, 6], [5, 9], [6, 8], [8, 9]]

perm: [7, 1, 0, 2, 4, 3, 9, 8, 5, 6]
  2-edge mismatch size: 26
  [[0, 1], [0, 2], [0, 5], [0, 6], [0, 7], [0, 8], [0, 9], [1, 2], [1, 3], [1, 7], [2, 3], [2, 4], [2, 5], [2, 9], [3, 4], [3, 5], [3, 6], [3, 7], [3, 9], [4, 5], [4, 6], [4, 8], [4, 9], [5, 6], [6, 7], [7, 9]]

perm: [5, 4, 2, 0, 1, 3, 6, 8, 7, 9]
  2-edge mismatch size: 24
  [[0, 1], [0, 2], [0, 3], [0, 7], [0, 9], [1, 2], [1, 5], [1, 8], [1, 9], [2, 4], [2, 5], [2, 7], [2, 8], [3, 5], [3, 6], [3, 7], [3, 9], [4, 7], [4, 9], [5, 6], [6, 7], [6, 8], [7, 9], [8, 9]]

perm: [2, 6, 3, 0, 9, 8, 1, 5, 7, 4]
  2-edge mismatch size: 26
  [[0, 1], [0, 2], [0, 4], [0, 5], [0, 6], [0, 7], [0, 9], [1, 3], [1, 7], [1, 8], [2, 3], [2, 6], [2, 7], [2, 8], [3, 4], [3, 5], [3, 8]

### LC-equivalence search for symmetries

In [17]:
def local_complement_graph(A, v):
    """Apply local complementation at vertex v to adjacency matrix A (GF2)."""
    A = A.copy()
    neighbors = [u for u in range(len(A)) if A[v, u]]
    for i in range(len(neighbors)):
        for j in range(i + 1, len(neighbors)):
            u, w = neighbors[i], neighbors[j]
            A[u, w] ^= 1
            A[w, u] ^= 1
    return A


def adj_to_2edges(A):
    n = A.shape[0]
    return frozenset(
        frozenset({i, j})
        for i in range(n) for j in range(i+1, n)
        if A[i, j]
    )


def permute_adj(A, perm):
    """Apply qubit permutation to adjacency matrix."""
    n = A.shape[0]
    A_new = np.zeros((n, n), dtype=np.uint8)
    for i in range(n):
        for j in range(n):
            A_new[perm[i], perm[j]] = A[i, j]
    return A_new


def lc_equivalent_up_to_perm(A_target, A_start, max_depth=6):
    """
    BFS over local complementation sequences to check if A_start can reach A_target.
    Returns the LC sequence if found, else None.
    """
    target_edges = adj_to_2edges(A_target)

    def to_key(M): return M.tobytes()

    queue = [(A_start.copy(), [])]
    visited = {to_key(A_start)}

    while queue:
        cur, seq = queue.pop(0)
        if adj_to_2edges(cur) == target_edges:
            return seq
        if len(seq) >= max_depth:
            continue
        for v in range(A_start.shape[0]):
            nxt = local_complement_graph(cur, v)
            k = to_key(nxt)
            if k not in visited:
                visited.add(k)
                queue.append((nxt, seq + [v]))

    return None


def logical_action(perm, C_new):
    C_list = list(C_new)
    C_set  = {w: i for i, w in enumerate(C_list)}
    action = []
    for word in C_list:
        permuted = "".join(word[perm[i]] for i in range(len(perm)))
        if permuted not in C_set:
            raise ValueError(f"Permutation does not preserve codespace: {word} -> {permuted}")
        action.append(C_set[permuted])
    return action


def is_identity_action(action):
    return all(action[i] == i for i in range(len(action)))


def codeword_to_logical(word, C_new, k):
    idx = C_new.index(word)
    return format(idx, f"0{k}b")


# Search valid_3edge candidates for nontrivial logical action + LC-graph equivalence
# Use part-2 cumulative adjacency as the graph target
A_target = A_p2

# The correct target is the BASE graph A (from CWS conversion of the prologue state)
# NOT A_p1 or A_p2, which don't exist as graph states

# For each candidate permutation p, check:
# 1. Does p preserve C?
# 2. Is permute_adj(A, p) LC-equivalent to A?
# (The CZ round-robin doesn't produce a new graph — it's absorbed into the stabilizer conjugation)

# Collect results cleanly in one pass with higher max_depth
print("Searching valid_3edge for nontrivial logical action + LC-graph equivalence...\n")

results_nontrivial = []
results_identity = []

for p in valid_3edge:
    if not check_codeword_preserving(p, C):
        continue

    action = logical_action(p, C)
    trivial = is_identity_action(action)

    A_permuted = permute_adj(A, p)
    lc_seq = lc_equivalent_up_to_perm(A, A_permuted, max_depth=6)  # 6 is enough

    if lc_seq is None:
        continue

    if not trivial:
        results_nontrivial.append((p, action, lc_seq))
        print(f"*** NONTRIVIAL FOUND ***")
        print(f"  perm={p}")
        print(f"  logical action={action}")
        print(f"  LC sequence={lc_seq}\n")
    else:
        results_identity.append((p, action, lc_seq))
        print(f"(stabilizer automorphism)")
        print(f"  perm={p}  LC={lc_seq}\n")

print(f"\n=== Done ===")
print(f"Nontrivial: {len(results_nontrivial)}")
print(f"Stabilizer auts: {len(results_identity)}")

Searching valid_3edge for nontrivial logical action + LC-graph equivalence...

*** NONTRIVIAL FOUND ***
  perm=[5, 9, 8, 7, 6, 2, 1, 0, 3, 4]
  logical action=[0, 2, 1, 3]
  LC sequence=[3, 4, 6, 9]

(stabilizer automorphism)
  perm=[2, 1, 0, 3, 4, 8, 9, 7, 5, 6]  LC=[3, 4, 5, 8]

(stabilizer automorphism)
  perm=[0, 1, 2, 3, 4, 7, 6, 8, 5, 9]  LC=[5, 6, 7, 9]

*** NONTRIVIAL FOUND ***
  perm=[8, 6, 5, 7, 9, 2, 1, 0, 3, 4]
  logical action=[0, 2, 1, 3]
  LC sequence=[3, 4, 6, 9, 5, 8]

*** NONTRIVIAL FOUND ***
  perm=[8, 9, 7, 5, 6, 3, 1, 2, 0, 4]
  logical action=[0, 2, 1, 3]
  LC sequence=[1, 2, 4, 2, 5, 8]

*** NONTRIVIAL FOUND ***
  perm=[8, 9, 7, 5, 6, 3, 4, 0, 2, 1]
  logical action=[0, 2, 1, 3]
  LC sequence=[2, 4, 5, 8]

*** NONTRIVIAL FOUND ***
  perm=[5, 9, 8, 7, 6, 0, 4, 3, 2, 1]
  logical action=[0, 2, 1, 3]
  LC sequence=[1, 4, 6, 9]

*** NONTRIVIAL FOUND ***
  perm=[7, 6, 5, 8, 9, 3, 1, 0, 2, 4]
  logical action=[0, 2, 1, 3]
  LC sequence=[1, 4, 0, 3, 8, 9]

*** NONTRIVIA

KeyboardInterrupt: 

### Full verification of a candidate symmetry

In [ ]:
# Edit p_candidate to whichever permutation you want to verify
# (e.g. the first nontrivial result, or any element of valid_3edge)
p_candidate = results[0][0] if results else valid_3edge[0] if valid_3edge else list(range(n))

A_permuted = permute_adj(A_target, p_candidate)
lc_seq = lc_equivalent_up_to_perm(A_target, A_permuted, max_depth=20)

print(f"Candidate perm: {p_candidate}")
print(f"Mismatch before LC: {sorted([sorted(e) for e in adj_to_2edges(A_permuted) ^ adj_to_2edges(A_target)])}")

if lc_seq is not None:
    A_lc = A_permuted.copy()
    for v in lc_seq:
        A_lc = local_complement_graph(A_lc, v)

    print(f"\n2-edges preserved after LC: {adj_to_2edges(A_lc) == adj_to_2edges(A_target)}")

    three_edges_orig = {e for e in hyperedges_p2 if len(e) == 3}
    three_edges_perm = {frozenset(p_candidate[q] for q in e) for e in three_edges_orig}
    print(f"3-edges preserved under p:   {three_edges_orig == three_edges_perm}")
    print(f"Codewords preserved under p: {check_codeword_preserving(p_candidate, C)}")

    print(f"\nLogical mapping (ZI=bit0, IZ=bit1 -> XI=bit0, IX=bit1):")
    for word in C:
        permuted = "".join(word[p_candidate[i]] for i in range(len(p_candidate)))
        print(f"  |{codeword_to_logical(word, C, k=2)}> -> |{codeword_to_logical(permuted, C, k=2)}>")
    print(f"\nLC sequence: complement at qubits {lc_seq}")
else:
    print("No LC sequence found within search depth.")